# Build the fsl training image - fast, one run

Reads project/region from `configs/pipeline_config.json` (no re-typing), then
builds and pushes the image in one cell. Default tag is **`candidate`** - the
tag the test pipeline uses. When a candidate passes the gate, the pipeline
retags it to `production`; the production pipeline uses `production`.

**Just run the two cells below.** Works whether this notebook sits in `scripts/`
or the project root - it finds the root automatically.

Requires: Docker on the machine (your Workbench has it), and a Dockerfile in
the project root (you already have it).

## 1. Setup - find root, read config, decide the tag

In [ ]:
import json
import os
from pathlib import Path


def find_project_root(start=None):
    """Walk up from CWD to find the folder with Dockerfile + configs/."""
    p = Path(start or Path.cwd()).resolve()
    for _ in range(6):
        if (p / "Dockerfile").exists() and (p / "configs" / "pipeline_config.json").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(
        "Couldn't find project root (needs Dockerfile + configs/pipeline_config.json). "
        "Run this from inside the project."
    )


ROOT = find_project_root()
os.chdir(ROOT)                       # docker build needs to run from the root (build context)
print("Project root:", ROOT)

with open("configs/pipeline_config.json") as f:
    CFG = json.load(f)

PROJECT = CFG["project"]
REGION  = CFG["region"]

# --- the one knob you might change ---
VERSION = "candidate"   # the tag the test pipeline uses; promoted to "production" on gate pass
REPO    = "fsl-images"
IMAGE   = "train"

IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT}/{REPO}/{IMAGE}:{VERSION}"

# guard: unedited placeholder project would produce a broken URI
if "YOUR-PROJECT" in PROJECT or PROJECT.endswith("-ID"):
    raise ValueError(
        f"config still has a placeholder project: {PROJECT!r}. "
        "Edit configs/pipeline_config_samld.json.json first."
    )

print("Will build and push:")
print(" ", IMAGE_URI)

## 2. Build + push (one run)

Ensures the Artifact Registry repo exists, authenticates Docker, builds, pushes.
The build step is the slow one (installs torch) - but Docker caches layers, so
if only `src/fsl/` changed since last time, it's fast; only a `pyproject.toml`
change triggers a full dependency reinstall.

In [ ]:
# ensure the Docker-format repo exists (harmless if it already does)
!gcloud artifacts repositories describe {REPO} --location={REGION} --project={PROJECT} >/dev/null 2>&1 || gcloud artifacts repositories create {REPO} --repository-format=docker --location={REGION} --project={PROJECT}

# let Docker authenticate to Artifact Registry (safe to re-run)
!gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet

# build (from ROOT, so the Dockerfile + src/ + scripts/ are the build context)
!docker build -t {IMAGE_URI} .

# push
!docker push {IMAGE_URI}

print()
print("Done. Image pushed:")
print(" ", IMAGE_URI)

## 3. (optional) Confirm it's there / paste into config

Lists the images in the repo so you can see the tag landed. If your
`serving_container_image_uri` or the pipeline's training image should point at
this, here's the exact line to paste into `configs/pipeline_config.json`.

In [ ]:
r = sh(["docker", "run", "--rm", "--entrypoint", "python", IMG, "scripts/al_simulation_entry.py", "--help"])
print("AL OK" if "--queue-size" in r.stdout else "AL BRAK - obraz bez nowego skryptu")

In [ ]:
import subprocess, textwrap

IMG = "us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate"
code = textwrap.dedent("""
    import io, pandas, pyarrow
    df = pandas.DataFrame({"a": [1.0, 2.0]})
    buf = io.BytesIO(); df.to_parquet(buf)
    back = pandas.read_parquet(io.BytesIO(buf.getvalue()))
    print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__,
          "| parquet roundtrip OK:", back.shape)
    from fsl.data.samld import GroupedEpisodeSampler, build_task_samplers
    from fsl.models.registry import get_encoder, known_encoders
    print("encoders", known_encoders())
""")
r = subprocess.run(["docker", "run", "--rm", "--entrypoint", "python", IMG, "-c", code],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

In [ ]:
!docker run --rm --entrypoint ls us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate scripts/

In [ ]:
!docker run --rm --entrypoint python \
  us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate \
  -c "import pyarrow; \
from fsl.data.samld import load_frozen_samld; \
from fsl.models.registry import known_encoders; \
from fsl.data.registry import get_input_geometry; \
print('encoders:', known_encoders(), '| samld:', get_input_geometry('samld'), '| pyarrow:', pyarrow.__version__)"

In [ ]:
!gcloud logging read 'resource.labels.job_id="8305495615408701440"' \
  --project dark-data-discovery --limit 300 \
  --format 'value(textPayload)' > /tmp/job.log
grep -B 3 -A 40 "Traceback\|ComponentError\|Error" /tmp/job.log | head -80

## Notes

- **Rebuild when code changes.** Because the pipeline uses this image (not
  inlined code), a change in `src/fsl/` only reaches the pipeline after you
  rerun cell 2. That's the trade for having the code in one place instead of
  duplicated across components.
- **`candidate` vs a version number.** `candidate` overwrites each build - it's
  "the current thing under test". If you want a permanent, non-overwriting tag
  (e.g. for a specific milestone), change `VERSION` to `v1`, `v2`, etc. The
  candidate/production flow doesn't need that, but it's there if you want it.
- **Speed.** First build is slow (torch). Later builds reuse cached layers
  unless `pyproject.toml` changed.